In [1]:
import pandas as pd
import glob
import os

In [13]:

input_path = './dataset_structured/01_tabular/poi_converted_wgs/splits/poi_converted_wgs_part_*.csv'
output_dir = './dataset_structured/05_processed'

if not os.path.exists(output_dir):
    os.makedirs(output_dir)

file_list = glob.glob(input_path)
print(f"Found {len(file_list)} POI parts.")

Found 9 POI parts.


In [9]:
include_mid_categories = {
    'transport': ['地铁', '公交站'],
    'toilet': ['公厕'],
    'medical_service': ['综合医院', '社区医疗', '诊所', '急救中心', '药店', '敬老院', '养老院', '福利院']
}

exclude_mid_categories = ['动物医疗', '宠物用品', '成人', '兽医']


In [10]:
all_chunks = []

for file in file_list:
    df = pd.read_csv(file)
    df[['名称', '大类', '中类']] = df[['名称', '大类', '中类']].fillna('')
    
    mask_transport = df['中类'].isin(include_mid_categories['transport'])
    mask_toilet = df['中类'].isin(include_mid_categories['toilet'])
    
    mask_medical = (
        df['中类'].str.contains('|'.join(include_mid_categories['medical_service'])) & 
        ~df['中类'].str.contains('|'.join(exclude_mid_categories))
    )
    
    filtered = df[mask_transport | mask_toilet | mask_medical].copy()
    
    def get_refined_type(row):
        if row['中类'] in include_mid_categories['toilet']: return 'Toilet'
        if row['中类'] in include_mid_categories['transport']: return 'Transport'
        return 'Medical_Service'

    filtered['category_group'] = filtered.apply(get_refined_type, axis=1)
    
    all_chunks.append(filtered)
    print(f"Processed {os.path.basename(file)}: {len(filtered)} high-quality POIs kept.")

Processed poi_converted_wgs_part_01.csv: 8494 high-quality POIs kept.
Processed poi_converted_wgs_part_02.csv: 5284 high-quality POIs kept.
Processed poi_converted_wgs_part_03.csv: 4063 high-quality POIs kept.
Processed poi_converted_wgs_part_04.csv: 3617 high-quality POIs kept.
Processed poi_converted_wgs_part_05.csv: 3585 high-quality POIs kept.
Processed poi_converted_wgs_part_06.csv: 3398 high-quality POIs kept.
Processed poi_converted_wgs_part_07.csv: 3393 high-quality POIs kept.
Processed poi_converted_wgs_part_08.csv: 4524 high-quality POIs kept.
Processed poi_converted_wgs_part_09.csv: 2320 high-quality POIs kept.


In [14]:
final_poi_df = pd.concat(all_chunks, ignore_index=True)
final_poi_df.drop_duplicates(subset=['lon_wgs', 'lat_wgs', '名称'], inplace=True)

final_poi_df.to_csv(f'{output_dir}/cleaned_target_pois.csv', index=False, encoding='utf_8_sig')
print(f"Success! Saved {len(final_poi_df)} POIs to '{output_dir}/cleaned_target_pois.csv'.")

Success! Saved 38677 POIs to './dataset_structured/05_processed/cleaned_target_pois.csv'.
